In [1]:
# datamodule.setup("fit")

In [10]:
import lightning as L
from hydra import compose, initialize
import awkward as ak
import numpy as np
import os
from omegaconf import DictConfig
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger  # , CometLogger

from lightning.pytorch.callbacks import TQDMProgressBar, ModelCheckpoint

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main")
from mltau.tools.io.preprocessed_ParTau_dataloader import ParTDataModule
# from mltau.models.ParTau_module import ParTauModule
from mltau.models import MultiParTau_module, SingleParTau_module
from mltau.tools.evaluation import inference

cfg.training.dataloader.batch_size = 128
cfg.dataset.data_dir = "/scratch/persistent/laurits/ml-tau/0412_increased_stats/"
print("cfg.dataset.data_dir:", cfg.dataset.data_dir)

cfg.dataset.data_dir: /scratch/persistent/laurits/ml-tau/0412_increased_stats/


In [11]:
cfg.training.model.name = "SingleParTau"

task_dir_map = {
    "is_tau": "isTau",
    "charge": "charge",
    "decay_mode": "DM",
    "kinematics": "kin",
}

single_task_base_output_dir = "/home/norman/ml-tau/test_inference_single"
single_task_models_dir = "/home/norman/0422"

In [12]:
def run_single_task_inference(task, best_ckpt_path=None, model_name="SingleParTau"):
    from mltau.tools.evaluation import inference

    if model_name != "SingleParTau":
        raise ValueError("This helper is only for SingleParTau.")

    cfg.training.model.name = model_name
    cfg.training.model.task = task
    cfg.output_dir = os.path.join(single_task_base_output_dir, task_dir_map[task])

    if best_ckpt_path is None:
        best_ckpt_path = os.path.join(
            single_task_models_dir,
            task_dir_map[task],
            "models",
            "ParT-model_best.ckpt",
        )

    print("cfg.output_dir:", cfg.output_dir)
    print("best_ckpt_path:", best_ckpt_path)

    if not os.path.exists(best_ckpt_path):
        print(f"[WARNING] Best checkpoint not found at {best_ckpt_path}. Skipping inference.")
        return

    print()
    print(f"[INFO] Running inference on test set using {best_ckpt_path}")
    best_model = SingleParTau_module.ParTauModule.load_from_checkpoint(
        best_ckpt_path,
        cfg=cfg,
        input_dim=17,
        num_dm_classes=6,
        task=task,
        map_location="cpu",
    )
    inference.create_predictions_files(
        best_model=best_model,
        model_name=model_name,
        cfg=cfg,
    )
    print("[INFO] SingleParTau prediction-file creation finished.")

In [5]:
task = "kinematics"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/kin
best_ckpt_path: /home/norman/0422/kin/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/kin/models/ParT-model_best.ckpt
[INFO] Prediction inputs:
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt


SingleParTau inference on z_test.pt:   0%|          | 0/2785 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/kin/predictions/z_test.parquet
[INFO] SingleParTau prediction-file creation finished.


In [6]:
task = "is_tau"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/isTau
best_ckpt_path: /home/norman/0422/isTau/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/isTau/models/ParT-model_best.ckpt
[INFO] Prediction inputs:
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/qq_test.pt
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt


SingleParTau inference on qq_test.pt:   0%|          | 0/18425 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/isTau/predictions/qq_test.parquet


SingleParTau inference on z_test.pt:   0%|          | 0/2785 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/isTau/predictions/z_test.parquet
[INFO] SingleParTau prediction-file creation finished.


In [7]:
task = "charge"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/charge
best_ckpt_path: /home/norman/0422/charge/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/charge/models/ParT-model_best.ckpt
[INFO] Prediction inputs:
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt


SingleParTau inference on z_test.pt:   0%|          | 0/2785 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/charge/predictions/z_test.parquet
[INFO] SingleParTau prediction-file creation finished.


In [8]:
task = "decay_mode"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/DM
best_ckpt_path: /home/norman/0422/DM/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/DM/models/ParT-model_best.ckpt
[INFO] Prediction inputs:
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt


SingleParTau inference on z_test.pt:   0%|          | 0/2785 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/DM/predictions/z_test.parquet
[INFO] SingleParTau prediction-file creation finished.


In [9]:
# Kinematics parquet sanity check: in-memory decode vs written parquet
import inspect
import awkward as ak
from mltau.tools.evaluation.inference import decode_kinematic_predictions
from mltau.tools.general import reinitialize_p4
import torch
from torch.utils.data import DataLoader
from mltau.tools.io.preprocessed_ParTau_dataloader import ParticleTransformerDataset
from mltau.tools.io.general import BatchInputs

if "shuffle" not in inspect.signature(ParticleTransformerDataset.__init__).parameters:
    raise RuntimeError(
        "ParticleTransformerDataset in this kernel does not support shuffle=False. "
        "Restart the kernel so the patched dataloader is imported before running this check."
    )

kin_parquet_path = os.path.join(single_task_base_output_dir, "kin", "predictions", "z_test.parquet")
print("kin_parquet_path:", kin_parquet_path)

kin_tensors = inference.load_tensors(os.path.join(cfg.dataset.data_dir, "z_test.pt"))
kin_dataset = ParticleTransformerDataset(
    kin_tensors,
    batch_size=cfg.training.dataloader.batch_size,
    shuffle=False,
)
kin_dataloader = DataLoader(kin_dataset, batch_size=None)
kin_batch = next(iter(kin_dataloader))
kin_inputs = BatchInputs(*kin_batch)

cfg.training.model.name = "SingleParTau"
cfg.training.model.task = "kinematics"
kin_best_ckpt_path = os.path.join(single_task_models_dir, "kin", "models", "ParT-model_best.ckpt")
kin_model = SingleParTau_module.ParTauModule.load_from_checkpoint(
    kin_best_ckpt_path,
    cfg=cfg,
    input_dim=17,
    num_dm_classes=6,
    task="kinematics",
    map_location="cpu",
)
kin_model.eval()

with torch.no_grad():
    kin_predictions, _, _ = kin_model.forward(kin_batch)

mem_pred_p4 = reinitialize_p4(
    decode_kinematic_predictions(kin_predictions["kinematics"], ak.Array(kin_inputs.reco_jet_p4s))
)
parquet = ak.from_parquet(kin_parquet_path)
parquet_pred_p4 = reinitialize_p4(parquet.tau_p4[: len(mem_pred_p4)])

print("first 5 in-memory pt:", ak.to_numpy(mem_pred_p4.pt[:5]))
print("first 5 parquet pt:", ak.to_numpy(parquet_pred_p4.pt[:5]))
print("first 5 in-memory eta:", ak.to_numpy(mem_pred_p4.eta[:5]))
print("first 5 parquet eta:", ak.to_numpy(parquet_pred_p4.eta[:5]))
print("first 5 in-memory phi:", ak.to_numpy(mem_pred_p4.phi[:5]))
print("first 5 parquet phi:", ak.to_numpy(parquet_pred_p4.phi[:5]))
print("max |pt diff| first batch:", np.max(np.abs(ak.to_numpy(mem_pred_p4.pt - parquet_pred_p4.pt))))
print("max |eta diff| first batch:", np.max(np.abs(ak.to_numpy(mem_pred_p4.eta - parquet_pred_p4.eta))))
print("max wrapped |phi diff| first batch:", np.max(np.abs(np.arctan2(np.sin(ak.to_numpy(mem_pred_p4.phi - parquet_pred_p4.phi)), np.cos(ak.to_numpy(mem_pred_p4.phi - parquet_pred_p4.phi))))))

kin_parquet_path: /home/norman/ml-tau/test_inference_single2/kin/predictions/z_test.parquet
first 5 in-memory pt: [36.150894   9.329726   5.723275  15.65682    4.0218563]
first 5 parquet pt: [36.150894   9.329726   5.723275  15.65682    4.0218563]
first 5 in-memory eta: [ 0.2910782 -1.3561839  0.8744515 -1.6206546 -1.68371  ]
first 5 parquet eta: [ 0.2910782 -1.3561839  0.8744515 -1.6206546 -1.68371  ]
first 5 in-memory phi: [ 2.4641216   1.2679007   0.14132635 -2.9237149  -1.4059001 ]
first 5 parquet phi: [ 2.4641216   1.2679007   0.14132635 -2.9237149  -1.4059001 ]
max |pt diff| first batch: 0.0
max |eta diff| first batch: 0.0
max wrapped |phi diff| first batch: 0.0


In [10]:
# Raw kinematics target/prediction diagnostics
import torch
from torch.utils.data import DataLoader
from mltau.tools.io.preprocessed_ParTau_dataloader import ParticleTransformerDataset
from mltau.tools.io.general import BatchInputs

kin_input_path = os.path.join(cfg.dataset.data_dir, "z_test.pt")
print("kin_input_path:", kin_input_path)

kin_tensors = inference.load_tensors(kin_input_path)
kin_dataset = ParticleTransformerDataset(
    kin_tensors,
    batch_size=cfg.training.dataloader.batch_size,
)
kin_dataloader = DataLoader(kin_dataset, batch_size=None)
kin_batch = next(iter(kin_dataloader))
kin_inputs = BatchInputs(*kin_batch)

cfg.training.model.name = "SingleParTau"
cfg.training.model.task = "kinematics"
kin_best_ckpt_path = os.path.join(single_task_models_dir, "kin", "models", "ParT-model_best.ckpt")
print("kin_best_ckpt_path:", kin_best_ckpt_path)

kin_model = SingleParTau_module.ParTauModule.load_from_checkpoint(
    kin_best_ckpt_path,
    cfg=cfg,
    input_dim=17,
    num_dm_classes=6,
    task="kinematics",
    map_location="cpu",
)
kin_model.eval()

with torch.no_grad():
    kin_predictions, kin_targets, _ = kin_model.forward(kin_batch)

raw_pred = kin_predictions["kinematics"].detach().cpu().numpy()
raw_target = kin_targets["kinematics"].detach().cpu().numpy()

print("raw_pred shape:", raw_pred.shape)
print("raw_target shape:", raw_target.shape)
print("first 5 raw predictions:")
print(raw_pred[:5])
print("first 5 raw targets:")
print(raw_target[:5])
print("median |pred - target| per component:")
print(np.median(np.abs(raw_pred - raw_target), axis=0))

kin_input_path: /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt
kin_best_ckpt_path: /home/norman/0422/kin/models/ParT-model_best.ckpt
raw_pred shape: (128, 5)
raw_target shape: (128, 5)
first 5 raw predictions:
[[-1.06922705e-02  2.40921415e-03 -1.62307918e-03  9.99595046e-01
  -1.51625760e-02]
 [ 1.50997955e-02  2.77004391e-03  1.72764808e-03  1.00034964e+00
  -2.38236570e+00]
 [-6.52856454e-02  3.15129384e-03 -1.26389787e-03  9.99948263e-01
  -5.77473566e-02]
 [ 3.72304209e-03  1.13936141e-03 -1.47793815e-03  9.99598384e-01
   9.22234729e-02]
 [-1.30202156e-02  2.13558413e-03 -1.26227736e-03  1.00027180e+00
  -2.77373381e-02]]
first 5 raw targets:
[[-2.7544848e-03 -4.6831369e-03 -6.7889685e-04  9.9999976e-01
   3.2035310e-02]
 [-1.2276537e-02 -1.7265141e-02 -1.5945025e-02  9.9987286e-01
  -2.1443834e+00]
 [-7.2839901e-02  1.8364191e-03 -1.7973781e-04  1.0000000e+00
  -3.8214941e-02]
 [-3.3401540e-03 -9.0003014e-06  2.6583672e-05  1.0000000e+00
   1.2651904e+00]
 [-2

In [8]:
# from tensorboard.backend.event_processing import event_accumulator

# log_dir = "/home/laurits/tmp/speedup_test2/tensorboard/ParTau_experiment/version_0/"

# ea = event_accumulator.EventAccumulator(log_dir)
# ea.Reload()

# # List available scalar tags
# print(ea.Tags()["scalars"])

# # Extract a specific scalar
# scalars = ea.Scalars("train_losses/decay_mode_loss")

# for s in scalars:
#     print(s.step, s.value)

## MultiParTau Inference

In [2]:
multi_task_base_output_dir = "/home/norman/ml-tau/test_inference_multi"
multi_task_best_ckpt_path = "/home/norman/ml-tau/test_inference/models/ParT-model_best-v4.ckpt"

In [3]:
def run_multi_task_inference(best_ckpt_path=None, output_dir=None):
    from mltau.tools.evaluation import inference

    cfg.training.model.name = "MultiParTau"
    cfg.dataset.data_dir = "/scratch/persistent/laurits/ml-tau/0412_increased_stats/"

    if output_dir is None:
        output_dir = multi_task_base_output_dir
    if best_ckpt_path is None:
        best_ckpt_path = multi_task_best_ckpt_path

    cfg.output_dir = output_dir
    print("cfg.dataset.data_dir:", cfg.dataset.data_dir)
    print("cfg.output_dir:", cfg.output_dir)
    print("best_ckpt_path:", best_ckpt_path)

    if not os.path.exists(best_ckpt_path):
        print(f"[WARNING] Best checkpoint not found at {best_ckpt_path}. Skipping inference.")
        return

    print()
    print(f"[INFO] Running inference on test set using {best_ckpt_path}")
    best_model = MultiParTau_module.ParTauModule.load_from_checkpoint(
        best_ckpt_path,
        cfg=cfg,
        input_dim=17,
        num_dm_classes=6,
        map_location="cpu",
        strict=False,
    )
    inference.create_predictions_files(
        best_model=best_model,
        model_name="MultiParTau",
        cfg=cfg,
    )
    print("[INFO] MultiParTau prediction-file creation finished.")

In [4]:
run_multi_task_inference()

cfg.dataset.data_dir: /scratch/persistent/laurits/ml-tau/0412_increased_stats/
cfg.output_dir: /home/norman/ml-tau/test_inference_multi
best_ckpt_path: /home/norman/ml-tau/test_inference/models/ParT-model_best-v4.ckpt

[INFO] Running inference on test set using /home/norman/ml-tau/test_inference/models/ParT-model_best-v4.ckpt


/opt/conda/lib/python3.11/site-packages/lightning/pytorch/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['ema_decay', 'ema_tagging_loss', 'ema_charge_loss', 'ema_dm_loss', 'ema_kin_loss']


[INFO] Prediction inputs:
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/qq_test.pt
 - /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt


MultiParTau inference on qq_test.pt:   0%|          | 0/18425 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_multi/predictions/qq_test.parquet


MultiParTau inference on z_test.pt:   0%|          | 0/2785 [00:00<?, ?batch/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_multi/predictions/z_test.parquet
[INFO] MultiParTau prediction-file creation finished.
